
# Coin Classification Base Model

This notebook adapts the lab exercise into a **five-class coin/object classifier**:

- `10_cent`
- `20_cent`
- `50_cent`
- `1_dollar`
- `unknown`

It trains a custom CNN using TensorFlow/Keras, evaluates it, saves the best model, and provides single-image inference with confidence-based unknown rejection.

> This notebook performs **classification of individual cropped objects**. Detecting and cropping multiple objects from a full table image will be added separately using OpenCV.



## Expected dataset structure

Prepare the dataset before running the notebook:

```text
coin_dataset/
├── train/
│   ├── 10_cent/
│   ├── 20_cent/
│   ├── 50_cent/
│   ├── 1_dollar/
│   └── unknown/
├── validation/
│   ├── 10_cent/
│   ├── 20_cent/
│   ├── 50_cent/
│   ├── 1_dollar/
│   └── unknown/
└── test/
    ├── 10_cent/
    ├── 20_cent/
    ├── 50_cent/
    ├── 1_dollar/
    └── unknown/
```

Each image should contain one main object. Front/back and old/new coin variations belong in the same denomination folder.


In [ ]:

# Uncomment when running in a fresh environment.
# %pip install -q tensorflow matplotlib scikit-learn pillow


In [1]:

from pathlib import Path
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


TensorFlow version: 2.21.0
GPU devices: []


## 1. Configuration

In [3]:

# Change this path to the location of your dataset.
DATASET_DIR = Path("dataset")

TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "validation"
TEST_DIR = DATASET_DIR / "test"

IMG_SIZE = (160, 160)
BATCH_SIZE = 3
SEED = 42
EPOCHS = 30

EXPECTED_CLASSES = ["10_cent", "20_cent", "50_cent", "1_dollar", "unknown"]

for directory in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if not directory.exists():
        raise FileNotFoundError(
            f"Missing folder: {directory.resolve()}\n"
            "Create the dataset structure shown above or update DATASET_DIR."
        )

print("Dataset folder:", DATASET_DIR.resolve())


Dataset folder: C:\Users\jiale\Downloads\MLAI project\dataset


## 2. Load training, validation and test data

In [4]:

train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED,
)

val_ds = keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)

test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)

print("Detected classes:", CLASS_NAMES)

if CLASS_NAMES != sorted(EXPECTED_CLASSES):
    print(
        "Warning: Keras sorts folder names alphabetically. "
        "The detected classes are valid as long as all five expected folders are present."
    )

missing = sorted(set(EXPECTED_CLASSES) - set(CLASS_NAMES))
if missing:
    raise ValueError(f"Missing expected class folders: {missing}")


Found 62 files belonging to 4 classes.
Found 11 files belonging to 4 classes.
Found 17 files belonging to 4 classes.
Detected classes: ['10_cent', '1_dollar', '20_cent', '50_cent']


ValueError: Missing expected class folders: ['unknown']

## 3. Inspect class balance

In [ ]:

def count_images_by_class(directory: Path):
    valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}
    counts = {}
    for class_name in CLASS_NAMES:
        class_dir = directory / class_name
        counts[class_name] = sum(
            1 for file in class_dir.rglob("*")
            if file.is_file() and file.suffix.lower() in valid_extensions
        )
    return counts

train_counts = count_images_by_class(TRAIN_DIR)
val_counts = count_images_by_class(VAL_DIR)
test_counts = count_images_by_class(TEST_DIR)

print("Training counts:", train_counts)
print("Validation counts:", val_counts)
print("Testing counts:", test_counts)

x = np.arange(len(CLASS_NAMES))
width = 0.25

plt.figure(figsize=(11, 5))
plt.bar(x - width, [train_counts[c] for c in CLASS_NAMES], width, label="Train")
plt.bar(x, [val_counts[c] for c in CLASS_NAMES], width, label="Validation")
plt.bar(x + width, [test_counts[c] for c in CLASS_NAMES], width, label="Test")
plt.xticks(x, CLASS_NAMES, rotation=25)
plt.ylabel("Number of images")
plt.title("Dataset class distribution")
plt.legend()
plt.tight_layout()
plt.show()


## 4. Display sample images

In [ ]:

plt.figure(figsize=(12, 8))

for images, labels in train_ds.take(1):
    number_to_show = min(15, len(images))
    for i in range(number_to_show):
        plt.subplot(3, 5, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        label_index = int(np.argmax(labels[i].numpy()))
        plt.title(CLASS_NAMES[label_index])
        plt.axis("off")

plt.tight_layout()
plt.show()



## 5. Data augmentation

Coins can appear at any rotation. The augmentation therefore includes full rotation, small translation, zoom, contrast and brightness changes.

Augmentation is applied only during training.


In [ ]:

data_augmentation = keras.Sequential(
    [
        layers.RandomRotation(factor=1.0),
        layers.RandomZoom(height_factor=(-0.10, 0.10), width_factor=(-0.10, 0.10)),
        layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
        layers.RandomContrast(factor=0.15),
        layers.RandomBrightness(factor=0.15, value_range=(0, 255)),
    ],
    name="coin_augmentation",
)

for images, _ in train_ds.take(1):
    sample = images[0]
    plt.figure(figsize=(10, 6))
    for i in range(8):
        augmented = data_augmentation(tf.expand_dims(sample, 0), training=True)
        plt.subplot(2, 4, i + 1)
        plt.imshow(tf.cast(augmented[0], tf.uint8))
        plt.axis("off")
    plt.tight_layout()
    plt.show()


## 6. Improve input performance

In [ ]:

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


## 7. Build the custom CNN base model

In [ ]:

def build_coin_cnn(input_shape=IMG_SIZE + (3,), num_classes=NUM_CLASSES):
    inputs = keras.Input(shape=input_shape, name="input_image")

    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255)(x)

    x = layers.Conv2D(32, 3, padding="same", activation="relu", name="conv1")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu", name="conv2")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu", name="conv3")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu", name="conv4")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu", name="conv5")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.30)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.40)(x)

    outputs = layers.Dense(num_classes, activation="softmax", name="classification")(x)

    model = keras.Model(inputs, outputs, name="coin_cnn_base")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_coin_cnn()
model.summary()


## 8. Train and save the best model

In [ ]:

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "coin_cnn_best.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=BEST_MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
]

start_time = time.perf_counter()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

training_time = time.perf_counter() - start_time
print(f"Training time: {training_time:.2f} seconds")
print("Best model saved to:", BEST_MODEL_PATH.resolve())


## 9. Plot accuracy and loss

In [ ]:

def plot_training_history(history):
    history_data = history.history
    epochs = range(1, len(history_data["loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history_data["accuracy"], label="Training accuracy")
    plt.plot(epochs, history_data["val_accuracy"], label="Validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training and validation accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history_data["loss"], label="Training loss")
    plt.plot(epochs, history_data["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and validation loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_training_history(history)


## 10. Evaluate on the test set

In [ ]:

best_model = keras.models.load_model(BEST_MODEL_PATH)

test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

probabilities = best_model.predict(test_ds, verbose=1)
predicted_indices = np.argmax(probabilities, axis=1)

true_indices = np.concatenate([
    np.argmax(labels.numpy(), axis=1)
    for _, labels in test_ds
])

print(
    classification_report(
        true_indices,
        predicted_indices,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)


## 11. Confusion matrix

In [ ]:

cm = confusion_matrix(true_indices, predicted_indices)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES,
)

fig, ax = plt.subplots(figsize=(9, 8))
display.plot(ax=ax, xticks_rotation=30, values_format="d")
plt.title("Coin classifier confusion matrix")
plt.tight_layout()
plt.show()



## 12. Single-image prediction with unknown rejection

The model already has an `unknown` class. A confidence threshold is also used so that uncertain predictions are rejected.

Tune this threshold using validation experiments rather than treating `0.75` as final.


In [ ]:

CONFIDENCE_THRESHOLD = 0.75

def predict_single_image(
    image_path,
    model=best_model,
    threshold=CONFIDENCE_THRESHOLD,
):
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    image = keras.utils.load_img(image_path, target_size=IMG_SIZE)
    image_array = keras.utils.img_to_array(image)
    batch = np.expand_dims(image_array, axis=0)

    probabilities = model.predict(batch, verbose=0)[0]
    class_index = int(np.argmax(probabilities))
    confidence = float(probabilities[class_index])
    predicted_label = CLASS_NAMES[class_index]

    if predicted_label == "unknown" or confidence < threshold:
        final_label = "unknown"
    else:
        final_label = predicted_label

    return {
        "label": final_label,
        "raw_prediction": predicted_label,
        "confidence": confidence,
        "probabilities": {
            class_name: float(probability)
            for class_name, probability in zip(CLASS_NAMES, probabilities)
        },
    }

# Example:
# result = predict_single_image("sample_images/test_coin.jpg")
# print(json.dumps(result, indent=2))


## 13. Display a prediction

In [ ]:

def show_prediction(image_path, threshold=CONFIDENCE_THRESHOLD):
    result = predict_single_image(image_path, threshold=threshold)

    image = keras.utils.load_img(image_path)

    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.title(
        f"{result['label']}\n"
        f"Confidence: {result['confidence']:.2%}"
    )
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    print(json.dumps(result, indent=2))

# Example:
# show_prediction("sample_images/test_coin.jpg")


## 14. Save class names and configuration

In [ ]:

config = {
    "class_names": CLASS_NAMES,
    "image_size": list(IMG_SIZE),
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "model_path": str(BEST_MODEL_PATH),
}

config_path = MODEL_DIR / "coin_model_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

print("Configuration saved to:", config_path.resolve())



## Next development stage

After this classifier works reliably on cropped individual objects:

1. Use OpenCV foreground segmentation or contour detection on a full table image.
2. Crop each detected object.
3. Pass every crop into `predict_single_image`.
4. Add only recognised coin values.
5. Display unknown objects without adding them to the total.
6. Integrate the pipeline into Streamlit.
